In [5]:
%cd "/nas/longleaf/home/amele/Bios 740/Final project/spert"
!pwd
!ls
import sys
!{sys.executable} -m pip install openai

/nas/longleaf/home/amele/Bios 740/Final project/spert
/nas/longleaf/home/amele/Bios 740/Final project/spert


/nas/longleaf/home/amele/.conda/envs/spert_env/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


ADKG.json	  configs      LICENSE	  __pycache__	    scripts
args.py		  data	       MDKG.json  README.md	    spert
config_reader.py  __init__.py  outputs	  requirements.txt  spert.py
  Using cached openai-2.35.1-py3-none-any.whl.metadata (31 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
Using cached openai-2.35.1-py3-none-any.whl (1.3 MB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 49.6 MB/s  0:00:00
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [openai]11/12 [openai]e]-types]


In [55]:
import os
import json
from typing import Dict, Any
from openai import OpenAI




client = OpenAI()



INPUT_PATH = "data/datasets/adkg/test.json"


OUTPUT_PATH = "outputs/gpt_annotations/adkg_gpt_test20.json"

MODEL = "gpt-4o-mini"

ALLOWED_ENTITY_TYPES = {"drug", "disease"}

ALLOWED_RELATION_TYPES = {
    "treatment_for",
    "causes",
    "associated_with",
    "no_relation"
}




def annotate_sentence(text: str, model: str = MODEL) -> Dict[str, Any]:
    prompt = f"""
You are extracting biomedical entities and relations from a sentence.

Return ONLY valid JSON.
Do not include markdown.
Do not include explanations.

Allowed entity types:
- drug
- disease

Important rules:
- Only use the allowed entity types above.
- Do NOT invent labels like gene, mutation, protein, method, physiology, or chemical.
- Even if a phrase seems biologically like a gene or mutation, you must only use the allowed dataset labels.
- Use exact character offsets from the sentence.
- start_char is inclusive.
- end_char is exclusive.
- The text field must exactly match text[start_char:end_char].

Allowed relation types:
- treatment_for
- causes
- associated_with
- no_relation

Sentence:
{text}

Return JSON in this exact format:
{{
  "entities": [
    {{
      "id": "T1",
      "type": "drug",
      "text": "entity text",
      "start_char": 0,
      "end_char": 10
    }}
  ],
  "relations": [
    {{
      "type": "associated_with",
      "head_id": "T1",
      "tail_id": "T2",
      "evidence": "short evidence phrase"
    }}
  ]
}}
"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You are a biomedical information extraction system. Return only valid JSON."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    content = response.choices[0].message.content.strip()

    
    if content.startswith("```json"):
        content = content.replace("```json", "").replace("```", "").strip()
    elif content.startswith("```"):
        content = content.replace("```", "").strip()

    return json.loads(content)




def convert_token_entities_to_char_entities(ex: Dict[str, Any]):
    """
    Converts dataset token-based entities into character-based entities.
    This is needed because GPT predicts character offsets.
    """
    tokens = ex["tokens"]
    text = " ".join(tokens)

    token_starts = []
    current = 0

    for tok in tokens:
        token_starts.append(current)
        current += len(tok) + 1

    converted_entities = []

    for idx, ent in enumerate(ex.get("entities", [])):
        start_token = ent["start"]
        end_token = ent["end"]

        start_char = token_starts[start_token]
        end_char = token_starts[end_token - 1] + len(tokens[end_token - 1])

        converted_entities.append({
            "id": ent.get("id", f"T{idx + 1}"),
            "type": ent["type"],
            "start": start_char,
            "end": end_char,
            "text": text[start_char:end_char]
        })

    return text, converted_entities


def get_text_and_gold_entities(ex: Dict[str, Any]):
    """
    Handles both possible dataset formats:
    1. already has text
    2. has tokens only
    """
    if "text" in ex:
        text = ex["text"]
        gold_entities = ex.get("entities", [])

    elif "tokens" in ex:
        text, gold_entities = convert_token_entities_to_char_entities(ex)

    else:
        raise KeyError(f"No text or tokens field found. Available keys: {list(ex.keys())}")

    return text, gold_entities




def filter_allowed_entity_types(annotation: Dict[str, Any],
                                allowed_types=ALLOWED_ENTITY_TYPES) -> Dict[str, Any]:
    """
    Removes predicted entities that do not use the dataset's allowed labels.
    Also removes relations connected to removed entities.
    """
    filtered_entities = []
    valid_ids = set()

    for ent in annotation.get("entities", []):
        if ent.get("type") in allowed_types:
            filtered_entities.append(ent)
            valid_ids.add(ent.get("id"))

    filtered_relations = []

    for rel in annotation.get("relations", []):
        if (
            rel.get("head_id") in valid_ids
            and rel.get("tail_id") in valid_ids
            and rel.get("type") in ALLOWED_RELATION_TYPES
        ):
            filtered_relations.append(rel)

    return {
        "entities": filtered_entities,
        "relations": filtered_relations
    }


def fix_entity_offsets(text: str, annotation: Dict[str, Any]) -> Dict[str, Any]:
    """
    Fixes entity offsets by searching for the predicted entity text
    inside the sentence.
    """
    fixed_entities = []

    for ent in annotation.get("entities", []):
        ent_text = ent.get("text", "")

        start = text.lower().find(ent_text.lower())

        if start != -1:
            end = start + len(ent_text)

            ent["start_char"] = start
            ent["end_char"] = end
            ent["offset_valid"] = text[start:end].lower() == ent_text.lower()
        else:
            ent["offset_valid"] = False

        fixed_entities.append(ent)

    valid_ids = {ent["id"] for ent in fixed_entities}

    fixed_relations = []

    for rel in annotation.get("relations", []):
        if rel.get("head_id") in valid_ids and rel.get("tail_id") in valid_ids:
            fixed_relations.append(rel)

    return {
        "entities": fixed_entities,
        "relations": fixed_relations
    }


def validate_offsets(text: str, annotation: Dict[str, Any]) -> Dict[str, Any]:
    """
    Keeps entities, but flags whether offsets are exactly valid.
    Does not delete entities just because offsets are slightly wrong.
    """
    valid_entities = []
    valid_ids = set()

    for ent in annotation.get("entities", []):
        start = ent.get("start_char")
        end = ent.get("end_char")
        ent_text = ent.get("text", "")

        offset_valid = (
            isinstance(start, int)
            and isinstance(end, int)
            and 0 <= start < end <= len(text)
            and text[start:end] == ent_text
        )

        ent["offset_valid"] = offset_valid
        valid_entities.append(ent)
        valid_ids.add(ent.get("id"))

    valid_relations = []

    for rel in annotation.get("relations", []):
        if rel.get("head_id") in valid_ids and rel.get("tail_id") in valid_ids:
            valid_relations.append(rel)

    return {
        "entities": valid_entities,
        "relations": valid_relations
    }



def run_pipeline(input_path=INPUT_PATH,
                 output_path=OUTPUT_PATH,
                 model=MODEL,
                 n_examples=20):

    with open(input_path, "r", encoding="utf-8") as f:
        examples = json.load(f)

    
    examples = examples[:n_examples]

    outputs = []

    for i, ex in enumerate(examples):
        text, gold_entities = get_text_and_gold_entities(ex)

        print(f"Annotating example {i + 1}/{len(examples)}")

        try:
            pred_raw = annotate_sentence(text, model=model)

            
            pred = filter_allowed_entity_types(pred_raw)
            pred = fix_entity_offsets(text, pred)
            pred = validate_offsets(text, pred)

            outputs.append({
                "doc_id": ex.get("doc_id"),
                "sent_id": ex.get("sent_id"),
                "text": text,
                "gold_entities": gold_entities,
                "gold_relations": ex.get("relations", []),
                "pred_entities": pred.get("entities", []),
                "pred_relations": pred.get("relations", []),
                "raw_pred": pred_raw,
                "error": None
            })

        except Exception as e:
            print("ERROR ON TEXT:")
            print(text)
            print("ERROR MESSAGE:")
            print(e)
            print("-" * 80)

            outputs.append({
                "doc_id": ex.get("doc_id"),
                "sent_id": ex.get("sent_id"),
                "text": text,
                "gold_entities": gold_entities,
                "gold_relations": ex.get("relations", []),
                "pred_entities": [],
                "pred_relations": [],
                "raw_pred": None,
                "error": str(e)
            })

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(outputs, f, indent=2)

    print(f"\nSaved annotations to: {output_path}")

    return outputs


if __name__ == "__main__":
    run_pipeline()

Annotating example 1/20
Annotating example 2/20
Annotating example 3/20
Annotating example 4/20
Annotating example 5/20
Annotating example 6/20
Annotating example 7/20
Annotating example 8/20
Annotating example 9/20
Annotating example 10/20
Annotating example 11/20
Annotating example 12/20
Annotating example 13/20
Annotating example 14/20
Annotating example 15/20
Annotating example 16/20
Annotating example 17/20
Annotating example 18/20
Annotating example 19/20
Annotating example 20/20

Saved annotations to: outputs/gpt_annotations/adkg_gpt_test20.json


In [42]:
import os, sys

os.environ["OPENAI_API_KEY"] = "PASTE_YOUR_NEW_KEY_HER"

!{sys.executable} scripts/gpt_annotation_pipeline.py

100%|███████████████████████████████████████████| 20/20 [01:49<00:00,  5.46s/it]
Saved 20 annotations to outputs/gpt_annotations/adkg_gpt_test20.json


In [56]:
import json

with open("outputs/gpt_annotations/adkg_gpt_test20.json", "r") as f:
    gpt_test20 = json.load(f)

gpt_test20[0]

{'doc_id': None,
 'sent_id': None,
 'text': "Alpha - amylase 1A copy number variants and the association with memory performance and Alzheimer ' s dementia .",
 'gold_entities': [{'id': 'T1',
   'type': 'drug',
   'start': 0,
   'end': 18,
   'text': 'Alpha - amylase 1A'},
  {'id': 'T2',
   'type': 'disease',
   'start': 88,
   'end': 110,
   'text': "Alzheimer ' s dementia"}],
 'gold_relations': [],
 'pred_entities': [{'id': 'T1',
   'type': 'disease',
   'text': "Alzheimer ' s dementia",
   'start_char': 88,
   'end_char': 110,
   'offset_valid': True}],
 'pred_relations': [],
 'raw_pred': {'entities': [{'id': 'T1',
    'type': 'disease',
    'text': "Alzheimer ' s dementia",
    'start_char': 88,
    'end_char': 110,
    'offset_valid': True}],
  'relations': []},
 'error': None}

In [57]:
def evaluate_entities(gold, pred, mode="strict"):
    tp = 0
    matched_gold = set()

    for p in pred:
        for i, g in enumerate(gold):
            if i in matched_gold:
                continue

            # gold uses start/end
            g_start = g["start"]
            g_end = g["end"]

            # pred uses start_char/end_char
            p_start = p["start_char"]
            p_end = p["end_char"]

            if mode == "strict":
                match = (
                    p_start == g_start and
                    p_end == g_end and
                    p["type"] == g["type"]
                )

            elif mode == "relaxed":
                overlap = p_start < g_end and p_end > g_start
                match = overlap and p["type"] == g["type"]

            if match:
                tp += 1
                matched_gold.add(i)
                break

    precision = tp / len(pred) if len(pred) > 0 else 0
    recall = tp / len(gold) if len(gold) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "gold": len(gold),
        "pred": len(pred),
        "correct": tp
    }

In [45]:
def evaluate_entities_dataset(examples, mode="strict"):
    total_correct = 0
    total_gold = 0
    total_pred = 0

    for ex in examples:
        result = evaluate_entities(
            gold=ex["gold_entities"],
            pred=ex["pred_entities"],
            mode=mode
        )

        total_correct += result["correct"]
        total_gold += result["gold"]
        total_pred += result["pred"]

    precision = total_correct / total_pred if total_pred > 0 else 0
    recall = total_correct / total_gold if total_gold > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "gold": total_gold,
        "pred": total_pred,
        "correct": total_correct
    }

In [58]:
strict_all = evaluate_entities_dataset(gpt, mode="strict")
relaxed_all = evaluate_entities_dataset(gpt, mode="relaxed")

print("Strict entity metrics:")
print(strict_all)

print("\nRelaxed entity metrics:")
print(relaxed_all)

Strict entity metrics:
{'precision': 0.3333333333333333, 'recall': 0.029850746268656716, 'f1': 0.05479452054794521, 'gold': 67, 'pred': 6, 'correct': 2}

Relaxed entity metrics:
{'precision': 0.3333333333333333, 'recall': 0.029850746268656716, 'f1': 0.05479452054794521, 'gold': 67, 'pred': 6, 'correct': 2}


In [61]:
import json
import pandas as pd




with open("outputs/gpt_annotations/adkg_gpt_test20.json", "r", encoding="utf-8") as f:
    gpt_test20 = json.load(f)



def safe_lower(x):
    return str(x).lower().strip()


def prf(correct, predicted, gold):
    precision = correct / predicted if predicted > 0 else 0
    recall = correct / gold if gold > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "correct": correct,
        "predicted": predicted,
        "gold": gold
    }


def spans_overlap(p_start, p_end, g_start, g_end):
    return p_start < g_end and p_end > g_start


def evaluate_entities(gold_entities, pred_entities, mode="strict"):
    """
    mode options:
    - strict: exact start, exact end, exact type
    - relaxed: span overlap + exact type
    - text_type: exact entity text + exact type, ignores offsets
    - text_only: exact entity text only, ignores type and offsets
    """

    correct = 0
    matched_gold = set()

    for p in pred_entities:
        for i, g in enumerate(gold_entities):
            if i in matched_gold:
                continue

            if mode == "strict":
                match = (
                    p.get("start_char") == g.get("start")
                    and p.get("end_char") == g.get("end")
                    and p.get("type") == g.get("type")
                )

            elif mode == "relaxed":
                p_start = p.get("start_char")
                p_end = p.get("end_char")
                g_start = g.get("start")
                g_end = g.get("end")

                match = (
                    isinstance(p_start, int)
                    and isinstance(p_end, int)
                    and isinstance(g_start, int)
                    and isinstance(g_end, int)
                    and spans_overlap(p_start, p_end, g_start, g_end)
                    and p.get("type") == g.get("type")
                )

            elif mode == "text_type":
                match = (
                    safe_lower(p.get("text")) == safe_lower(g.get("text"))
                    and p.get("type") == g.get("type")
                )

            elif mode == "text_only":
                match = safe_lower(p.get("text")) == safe_lower(g.get("text"))

            else:
                raise ValueError("mode must be strict, relaxed, text_type, or text_only")

            if match:
                correct += 1
                matched_gold.add(i)
                break

    return prf(
        correct=correct,
        predicted=len(pred_entities),
        gold=len(gold_entities)
    )


def evaluate_entities_dataset(examples, mode="strict"):
    total_correct = 0
    total_predicted = 0
    total_gold = 0

    for ex in examples:
        result = evaluate_entities(
            gold_entities=ex.get("gold_entities", []),
            pred_entities=ex.get("pred_entities", []),
            mode=mode
        )

        total_correct += result["correct"]
        total_predicted += result["predicted"]
        total_gold += result["gold"]

    return prf(
        correct=total_correct,
        predicted=total_predicted,
        gold=total_gold
    )


def entity_signature_by_id(entities, id_key="id", mode="strict"):
    """
    Creates a dictionary mapping entity id to a comparable signature.

    For predicted entities:
    - uses start_char/end_char/type/text

    For gold entities:
    - uses start/end/type/text
    """

    out = {}

    for ent in entities:
        ent_id = ent.get(id_key)

        if ent_id is None:
            continue

        if mode == "strict":
            start = ent.get("start", ent.get("start_char"))
            end = ent.get("end", ent.get("end_char"))
            out[ent_id] = (
                start,
                end,
                ent.get("type")
            )

        elif mode == "text_type":
            out[ent_id] = (
                safe_lower(ent.get("text")),
                ent.get("type")
            )

        elif mode == "text_only":
            out[ent_id] = safe_lower(ent.get("text"))

        else:
            raise ValueError("mode must be strict, text_type, or text_only")

    return out


def evaluate_relations(gold_entities, pred_entities, gold_relations, pred_relations, mode="strict"):
    """
    Relation is correct if:
    - head entity matches
    - tail entity matches
    - relation type matches

    mode options:
    - strict: head/tail matched by exact offsets + entity type
    - text_type: head/tail matched by entity text + entity type
    - text_only: head/tail matched by entity text only
    """

    gold_ent_map = entity_signature_by_id(gold_entities, mode=mode)
    pred_ent_map = entity_signature_by_id(pred_entities, mode=mode)

    gold_relation_set = set()

    for rel in gold_relations:
        head_id = rel.get("head_id")
        tail_id = rel.get("tail_id")

        if head_id in gold_ent_map and tail_id in gold_ent_map:
            gold_relation_set.add((
                gold_ent_map[head_id],
                rel.get("type"),
                gold_ent_map[tail_id]
            ))

    pred_relation_set = set()

    for rel in pred_relations:
        head_id = rel.get("head_id")
        tail_id = rel.get("tail_id")

        if head_id in pred_ent_map and tail_id in pred_ent_map:
            pred_relation_set.add((
                pred_ent_map[head_id],
                rel.get("type"),
                pred_ent_map[tail_id]
            ))

    correct = len(gold_relation_set.intersection(pred_relation_set))

    return prf(
        correct=correct,
        predicted=len(pred_relation_set),
        gold=len(gold_relation_set)
    )


def evaluate_relations_dataset(examples, mode="strict"):
    total_correct = 0
    total_predicted = 0
    total_gold = 0

    for ex in examples:
        result = evaluate_relations(
            gold_entities=ex.get("gold_entities", []),
            pred_entities=ex.get("pred_entities", []),
            gold_relations=ex.get("gold_relations", []),
            pred_relations=ex.get("pred_relations", []),
            mode=mode
        )

        total_correct += result["correct"]
        total_predicted += result["predicted"]
        total_gold += result["gold"]

    return prf(
        correct=total_correct,
        predicted=total_predicted,
        gold=total_gold
    )


results = []

metric_settings = [
    ("Entity Strict", "entity", "strict"),
    ("Entity Relaxed Overlap", "entity", "relaxed"),
    ("Entity Text + Type", "entity", "text_type"),
    ("Entity Text Only", "entity", "text_only"),
    ("Relation Strict", "relation", "strict"),
    ("Relation Text + Type", "relation", "text_type"),
    ("Relation Text Only", "relation", "text_only")
]

for metric_name, metric_level, mode in metric_settings:
    if metric_level == "entity":
        r = evaluate_entities_dataset(gpt_test20, mode=mode)
    else:
        r = evaluate_relations_dataset(gpt_test20, mode=mode)

    results.append({
        "System": "GPT-4o-mini + post-processing",
        "Metric": metric_name,
        "Precision": round(r["precision"], 4),
        "Recall": round(r["recall"], 4),
        "F1": round(r["f1"], 4),
        "Correct": r["correct"],
        "Predicted": r["predicted"],
        "Gold": r["gold"]
    })

results_df = pd.DataFrame(results)

results_df

,System,Metric,Precision,Recall,F1,Correct,Predicted,Gold
0,GPT-4o-mini + post-processing,Entity Strict,0.7273,0.2388,0.3596,16,22,67
1,GPT-4o-mini + post-processing,Entity Relaxed Overlap,0.7727,0.2537,0.3820,17,22,67
2,GPT-4o-mini + post-processing,Entity Text + Type,0.7727,0.2537,0.3820,17,22,67
3,GPT-4o-mini + post-processing,Entity Text Only,0.9091,0.2985,0.4494,20,22,67
4,GPT-4o-mini + post-processing,Relation Strict,0.0000,0.0000,0.0000,0,14,0
5,GPT-4o-mini + post-processing,Relation Text + Type,0.0000,0.0000,0.0000,0,14,0
6,GPT-4o-mini + post-processing,Relation Text Only,0.0000,0.0000,0.0000,0,14,0


In [64]:
import json
import pandas as pd


GOLD_PATH = "data/datasets/adkg/test.json"

BASELINE_PATH = "data/log/adkg_eval/2026-05-06_22:37:58.509020/predictions_test_epoch_0.json"



with open(GOLD_PATH, "r", encoding="utf-8") as f:
    gold_raw = json.load(f)

with open(BASELINE_PATH, "r", encoding="utf-8") as f:
    baseline_raw = json.load(f)

print("Gold type:", type(gold_raw))
print("Gold length:", len(gold_raw))
print("Gold first keys:", gold_raw[0].keys())

print("\nBaseline type:", type(baseline_raw))
print("Baseline length:", len(baseline_raw))
print("Baseline first keys:", baseline_raw[0].keys())

print("\nFirst baseline example:")
print(json.dumps(baseline_raw[0], indent=2)[:2000])

Gold type: <class 'list'>
Gold length: 1220
Gold first keys: dict_keys(['tokens', 'entities', 'relations', 'orig_id'])

Baseline type: <class 'list'>
Baseline length: 1220
Baseline first keys: dict_keys(['tokens', 'entities', 'relations'])

First baseline example:
{
  "tokens": [
    "Alpha",
    "-",
    "amylase",
    "1A",
    "copy",
    "number",
    "variants",
    "and",
    "the",
    "association",
    "with",
    "memory",
    "performance",
    "and",
    "Alzheimer",
    "'",
    "s",
    "dementia",
    "."
  ],
  "entities": [
    {
      "type": "other",
      "start": 11,
      "end": 13
    },
    {
      "type": "disease",
      "start": 14,
      "end": 18
    }
  ],
  "relations": []
}


In [69]:

def tokens_to_text_and_starts(tokens):
    text = " ".join(tokens)

    token_starts = []
    current = 0

    for tok in tokens:
        token_starts.append(current)
        current += len(tok) + 1

    return text, token_starts


def convert_entities_token_to_char(entities, tokens):
    text, token_starts = tokens_to_text_and_starts(tokens)

    converted = []

    for i, ent in enumerate(entities):
        start_token = ent.get("start")
        end_token = ent.get("end")

        if start_token is None or end_token is None:
            continue

        if start_token < 0 or end_token > len(tokens) or start_token >= end_token:
            continue

        start_char = token_starts[start_token]
        end_char = token_starts[end_token - 1] + len(tokens[end_token - 1])

        converted.append({
            "id": ent.get("id", f"T{i + 1}"),
            "type": ent.get("type"),
            "start": start_char,
            "end": end_char,
            "start_char": start_char,
            "end_char": end_char,
            "text": text[start_char:end_char]
        })

    return converted


def convert_relations(relations):
    """
    Converts relation head/tail fields into head_id/tail_id format.

    Handles common formats:
    - head_id / tail_id
    - head / tail as entity indices
    """
    converted = []

    for rel in relations:
        rel_type = rel.get("type")

        if "head_id" in rel and "tail_id" in rel:
            head_id = rel.get("head_id")
            tail_id = rel.get("tail_id")

        elif "head" in rel and "tail" in rel:
            head_id = f"T{rel.get('head') + 1}"
            tail_id = f"T{rel.get('tail') + 1}"

        else:
            continue

        converted.append({
            "type": rel_type,
            "head_id": head_id,
            "tail_id": tail_id
        })

    return converted


def get_pred_entities_from_baseline(ex):
    """
    Tries several possible keys because baseline prediction files
    can store predictions under different names.
    """
    for key in ["pred_entities", "predicted_entities", "entities"]:
        if key in ex:
            return ex[key]

    return []


def get_pred_relations_from_baseline(ex):
    """
    Tries several possible keys because baseline prediction files
    can store predictions under different names.
    """
    for key in ["pred_relations", "predicted_relations", "relations"]:
        if key in ex:
            return ex[key]

    return []


def make_baseline_eval_examples(gold_raw, baseline_raw, n_examples=20):
    """
    Combines gold ADKG test examples with SpERT baseline predictions.
    Output format matches the GPT output format.
    """
    examples = []

    gold_raw = gold_raw[:n_examples]
    baseline_raw = baseline_raw[:n_examples]

    for idx, gold_ex in enumerate(gold_raw):
        base_ex = baseline_raw[idx]

        tokens = gold_ex["tokens"]
        text, token_starts = tokens_to_text_and_starts(tokens)

        gold_entities = convert_entities_token_to_char(
            gold_ex.get("entities", []),
            tokens
        )

        pred_entities = convert_entities_token_to_char(
            get_pred_entities_from_baseline(base_ex),
            tokens
        )

        gold_relations = convert_relations(gold_ex.get("relations", []))
        pred_relations = convert_relations(get_pred_relations_from_baseline(base_ex))

        examples.append({
            "doc_id": gold_ex.get("doc_id", gold_ex.get("orig_id", idx)),
            "sent_id": gold_ex.get("sent_id", gold_ex.get("orig_id", idx)),
            "text": text,
            "gold_entities": gold_entities,
            "gold_relations": gold_relations,
            "pred_entities": pred_entities,
            "pred_relations": pred_relations
        })

    return examples


baseline_test20 = make_baseline_eval_examples(
    gold_raw=gold_raw,
    baseline_raw=baseline_raw,
    n_examples=20
)

print("Example baseline converted:")
print(json.dumps(baseline_test20[0], indent=2)[:2000])

Example baseline converted:
{
  "doc_id": "33220711_s0",
  "sent_id": "33220711_s0",
  "text": "Alpha - amylase 1A copy number variants and the association with memory performance and Alzheimer ' s dementia .",
  "gold_entities": [
    {
      "id": "T1",
      "type": "drug",
      "start": 0,
      "end": 18,
      "start_char": 0,
      "end_char": 18,
      "text": "Alpha - amylase 1A"
    },
    {
      "id": "T2",
      "type": "disease",
      "start": 88,
      "end": 110,
      "start_char": 88,
      "end_char": 110,
      "text": "Alzheimer ' s dementia"
    }
  ],
  "gold_relations": [],
  "pred_entities": [
    {
      "id": "T1",
      "type": "other",
      "start": 65,
      "end": 83,
      "start_char": 65,
      "end_char": 83,
      "text": "memory performance"
    },
    {
      "id": "T2",
      "type": "disease",
      "start": 88,
      "end": 110,
      "start_char": 88,
      "end_char": 110,
      "text": "Alzheimer ' s dementia"
    }
  ],
  "pred_relations"

In [66]:

baseline_results = []

metric_settings = [
    ("Entity Strict", "entity", "strict"),
    ("Entity Relaxed Overlap", "entity", "relaxed"),
    ("Entity Text + Type", "entity", "text_type"),
    ("Entity Text Only", "entity", "text_only"),
    ("Relation Strict", "relation", "strict"),
    ("Relation Text + Type", "relation", "text_type"),
    ("Relation Text Only", "relation", "text_only")
]

for metric_name, metric_level, mode in metric_settings:
    if metric_level == "entity":
        r = evaluate_entities_dataset(baseline_test20, mode=mode)
    else:
        r = evaluate_relations_dataset(baseline_test20, mode=mode)

    baseline_results.append({
        "System": "Baseline SpERT Model",
        "Metric": metric_name,
        "Precision": round(r["precision"], 4),
        "Recall": round(r["recall"], 4),
        "F1": round(r["f1"], 4),
        "Correct": r["correct"],
        "Predicted": r["predicted"],
        "Gold": r["gold"]
    })

baseline_results_df = pd.DataFrame(baseline_results)

baseline_results_df

,System,Metric,Precision,Recall,F1,Correct,Predicted,Gold
0,Baseline SpERT Model,Entity Strict,0.7324,0.7761,0.7536,52,71,67
1,Baseline SpERT Model,Entity Relaxed Overlap,0.7887,0.8358,0.8116,56,71,67
2,Baseline SpERT Model,Entity Text + Type,0.7324,0.7761,0.7536,52,71,67
3,Baseline SpERT Model,Entity Text Only,0.8028,0.8507,0.8261,57,71,67
4,Baseline SpERT Model,Relation Strict,0.2857,0.4615,0.3529,6,21,13
5,Baseline SpERT Model,Relation Text + Type,0.2857,0.4615,0.3529,6,21,13
6,Baseline SpERT Model,Relation Text Only,0.2857,0.4615,0.3529,6,21,13


In [67]:
combined_results_df = pd.concat(
    [baseline_results_df, results_df],
    ignore_index=True
)

combined_results_df

,System,Metric,Precision,Recall,F1,Correct,Predicted,Gold
0,Baseline SpERT Model,Entity Strict,0.7324,0.7761,0.7536,52,71,67
1,Baseline SpERT Model,Entity Relaxed Overlap,0.7887,0.8358,0.8116,56,71,67
2,Baseline SpERT Model,Entity Text + Type,0.7324,0.7761,0.7536,52,71,67
3,Baseline SpERT Model,Entity Text Only,0.8028,0.8507,0.8261,57,71,67
4,Baseline SpERT Model,Relation Strict,0.2857,0.4615,0.3529,6,21,13
5,Baseline SpERT Model,Relation Text + Type,0.2857,0.4615,0.3529,6,21,13
6,Baseline SpERT Model,Relation Text Only,0.2857,0.4615,0.3529,6,21,13
7,GPT-4o-mini + post-processing,Entity Strict,0.7273,0.2388,0.3596,16,22,67
8,GPT-4o-mini + post-processing,Entity Relaxed Overlap,0.7727,0.2537,0.3820,17,22,67
9,GPT-4o-mini + post-processing,Entity Text + Type,0.7727,0.2537,0.3820,17,22,67
